# Bank Distress Early-Warning Model: Modeling
**MSDS 696 Data Science Practicum II · Oussama Ennaciri**

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED = Path("..") / "data" / "processed"

panel = pd.read_parquet(PROCESSED / "panel_trend.parquet")
print(f"panel_trend: {panel.shape[0]:,} rows x {panel.shape[1]} cols")

panel_trend: 1,258,888 rows x 124 cols


### Capital headroom: the untrained benchmark

The rule says a bank must hold at least 8% total risk-based capital (and meet three other
minimums). A bank at 12% has four points of room; a bank at 8.5% has half a point. Less
room means closer to crossing the line.

**Headroom = the smallest gap between any of a bank's capital ratios and its minimum.**
The smallest gap is used because that is how the regulation itself works, the worst ratio
sets the capital tier, so it is the binding constraint.

The applicable ratios change in 2015 (Basel III adds common equity tier 1 and raises the
Tier 1 minimum), so the thresholds switch with the date, matching the label built in
`feature_selection.ipynb`. Ratios that do not apply in a given era contribute nothing.

It is built here, before the split, so the benchmark travels with the test rows. It is a
benchmark and not a feature, so the guardrail below excludes it from the model inputs.

In [2]:
# Minimums for "adequately capitalized": below any of these is undercapitalized.
# Source: 12 CFR 325.103 (1990-2014) and 12 CFR 324.403 (2015+), per
# ../literature/pca_label_definition.md
THRESHOLDS = {
    "pre2015": {"RBCRWAJ": 8.0, "RBC1RWAJ": 4.0, "RBC1AAJ": 4.0},
    "basel3":  {"RBCRWAJ": 8.0, "RBC1RWAJ": 6.0, "RBC1AAJ": 4.0, "RBCT1CER": 4.5},
}
BASEL3_START = pd.Timestamp("2015-01-01")

pre_basel3 = panel["REPDTE"] < BASEL3_START
gaps = pd.DataFrame(index=panel.index)

for ratio in ["RBCRWAJ", "RBC1RWAJ", "RBC1AAJ", "RBCT1CER"]:
    # A ratio absent from a regime's dict gets a NaN threshold, so its gap is NaN and
    # min() skips it -- e.g. CET1 simply does not exist before 2015.
    limit = np.where(pre_basel3,
                     THRESHOLDS["pre2015"].get(ratio, np.nan),
                     THRESHOLDS["basel3"].get(ratio, np.nan))
    gaps[ratio] = panel[ratio] - limit

panel["headroom"] = gaps.min(axis=1)

print(f"headroom: {panel['headroom'].isna().mean() * 100:.2f}% missing")
print(panel["headroom"].describe(percentiles=[.05, .25, .5, .75]).round(2).to_string())

headroom: 0.00% missing
count    1258888.00
mean           7.07
std           19.53
min       -16637.63
5%             1.83
25%            3.70
50%            5.29
75%            7.52
max          947.11


## Leakage guardrail

The target (`onset_4q`) asks whether a healthy bank falls to undercapitalized within the
**next four quarters**. That forward look is what makes this problem leak-prone: any column
or any split that lets a training row see past its own prediction quarter hands the model
the answer, and the score comes back beautiful and meaningless.

This section fixes the rules **once**, before a single model is fit:

1. which columns are allowed to be inputs,
2. how the data is cut by time,
3. assertions that fail loudly if either rule is broken.

Everything below this point can then use `FEATURES`, `train`, and `test` without
re-checking leakage each time.

### 1 · Columns that can never be inputs

Four groups have to come out of the feature list, for four different reasons:

| Group | Columns | Why it can't be an input |
|---|---|---|
| **Identifiers** | `CERT`, `REPDTE`, `NAME`, `ESTYMD` | Not predictors, a certificate number carries no risk information. Kept in the table for reference and for the time split |
| **Label parts** | `onset_4q`, `quarters_to_onset`, `pca_tier`, `is_distressed`, `is_healthy` | These *are* the answer, or what the answer was built from. `quarters_to_onset` literally records how soon distress arrived |
| **Retroactive macro** | `USREC` | The NBER recession flag is dated **after the fact**, the Dec 2007 recession start was announced Dec 2008. A `1` sitting in a 2008Q1 row is knowledge nobody had that quarter, arriving exactly when failures spike |
| **Future-built flags** | `near_merge_exit`, `fast_failure` | Both were built in `cleaning.ipynb` from outcomes only known afterward. `near_merge_exit` is positive **0.00%** of the time against a 0.74% baseline, a free "this bank is safe" signal |

Dropping `USREC` costs nothing: the other macro columns (rates, unemployment, credit
spreads, financial stress) carry the same business-cycle signal and *were* genuinely
published at the time.

**`roa_artifact` stays in.** It's computed from the current quarter's ROA only, so it looks
backward, not forward. It is the one flag that is safe as an input.

The two dropped flags are **not deleted**, they stay in `panel` so results can be reported
separately on the hard cases (`fast_failure`) and sensitivity checked both ways
(`near_merge_exit`), exactly as `cleaning.ipynb` intended.

In [3]:
# --- The drop list: one definition, used everywhere below -----------------------

# Identifiers and raw dates. Not predictors, but needed for the split and for reference.
ID_COLS = ["CERT", "REPDTE", "NAME", "ESTYMD"]

# The target and everything it was derived from.
LABEL_COLS = ["onset_4q", "quarters_to_onset", "pca_tier", "is_distressed", "is_healthy"]

# Flags built from information that only exists after the prediction quarter.
FUTURE_FLAGS = ["near_merge_exit", "fast_failure"]

# Macro series dated retroactively. NBER announces recession dates months to a year
# after they begin, so this column wasn't knowable in the quarter it marks.
LOOKAHEAD_MACRO = ["USREC"]

DROP_FROM_FEATURES = ID_COLS + LABEL_COLS + FUTURE_FLAGS + LOOKAHEAD_MACRO

TARGET = "onset_4q"
FEATURES = [c for c in panel.columns if c not in DROP_FROM_FEATURES]

# Text columns that will need encoding before any model that can't take strings.
CATEGORICAL = [c for c in FEATURES if str(panel[c].dtype) in ("object", "category")]

print(f"features kept:   {len(FEATURES)}")
print(f"columns dropped: {len(DROP_FROM_FEATURES)}  ->  {DROP_FROM_FEATURES}")
print(f"needs encoding:  {CATEGORICAL}")
print(f"roa_artifact kept as a feature: {'roa_artifact' in FEATURES}")

features kept:   113
columns dropped: 12  ->  ['CERT', 'REPDTE', 'NAME', 'ESTYMD', 'onset_4q', 'quarters_to_onset', 'pca_tier', 'is_distressed', 'is_healthy', 'near_merge_exit', 'fast_failure', 'USREC']
needs encoding:  ['STALP', 'BKCLASS', 'REGAGNT']
roa_artifact kept as a feature: True


### 2 · Split by time, with a gap year

**Why not a random split.** Shuffling rows puts 2008 and 2023 on both sides of the line.
The model trains on the same crisis it is later tested on and scores near-perfectly on
nothing at all.

**Why a gap year.** A training row at 2015Q4 is labeled using outcomes through 2016Q4.
If the test period opened in 2016, that training label would already describe the test
period. The gap is set to a full **four quarters**, exactly the label's horizon, so
every training label resolves before the test window opens.

**Why the test set stops at 2025Q1.** The panel runs to 2026Q1, but a row needs four
following quarters to be labeled at all. From 2025Q2 on, `cleaning.ipynb` blanked the
unobservable negatives (concern #8) and kept only the positives that had already crossedleaving **12 rows that are 100% positive with no negatives at all**. Left in the test set
they quietly inflate every recall figure. 2025Q1 is the last fully observable quarter.

Those trailing quarters aren't wasted: they keep every predictor, so they become the
**"who looks risky right now"** scoring set for the final presentation, a demo outputnot a validated score.

| Window | Dates | Purpose |
|---|---|---|
| **Train** | 1990Q1 – 2015Q4 | Fit the model |
| **Gap** | 2016Q1 – 2016Q4 | Buffer. Never fit on, never scored |
| **Test** | 2017Q1 – 2025Q1 | Honest evaluation, includes the 2023 failures |
| **Score now** | 2025Q2 – 2026Q1 | Unlabelable. Demo predictions only |

In [4]:
# --- Time boundaries ------------------------------------------------------------

TRAIN_END  = "2015-12-31"   # last quarter used for fitting
GAP_END    = "2016-12-31"   # 4 quarters wide = the label's forward horizon
TEST_START = "2017-01-01"
TEST_END   = "2025-03-31"   # last quarter with a full 4-quarter outcome window

# Only rows that carry a real label can be trained or scored on. Blank onset_4q means
# either "already distressed" or "outcome window not observable": neither is trainable.
labeled = panel[panel[TARGET].notna()]

train = labeled[labeled["REPDTE"] <= TRAIN_END]
test  = labeled[(labeled["REPDTE"] >= TEST_START) & (labeled["REPDTE"] <= TEST_END)]

# Held out on purpose. Exists to be excluded, not used.
gap = labeled[(labeled["REPDTE"] > TRAIN_END) & (labeled["REPDTE"] <= GAP_END)]

# No label yet, but every predictor is present -> the live "who looks risky now" set.
score_now = panel[(panel["REPDTE"] > TEST_END) & (panel[TARGET].isna())]

for name, df in [("train", train), ("gap", gap), ("test", test)]:
    print(f"{name:<6} {len(df):>10,} rows   "
          f"{df['REPDTE'].min().date()} to {df['REPDTE'].max().date()}   "
          f"{int(df[TARGET].sum()):>5,} positives ({df[TARGET].mean() * 100:.2f}%)")

print(f"{'score':<6} {len(score_now):>10,} rows   "
      f"{score_now['REPDTE'].min().date()} to {score_now['REPDTE'].max().date()}   "
      f"unlabeled (demo only)")

train   1,026,797 rows   1990-03-31 to 2015-12-31   8,414 positives (0.82%)
gap        24,181 rows   2016-03-31 to 2016-12-31      32 positives (0.13%)
test      166,754 rows   2017-03-31 to 2025-03-31     160 positives (0.10%)
score      17,698 rows   2025-06-30 to 2026-03-31   unlabeled (demo only)


### 3 · Assertions

The rules above are only worth something if breaking them is noisy. Each check below
corresponds to a specific way this project could leak:

1. **No leaky column survived**, catches a future-built flag sliding back into `FEATURES`
   after an edit upstream.
2. **Train ends before test begins**, catches an accidental shuffle or a bad date string.
3. **Training labels resolve before the test window**, the gap-year check, stated in terms
   of the label horizon rather than trusting the dates by eye.
4. **The rare-event rate survives in both halves**, catches rebalancing (`SMOTE`, class
   weights) applied before the split instead of to the training fold only.
5. **Every test quarter contains real negatives**, catches the positives-only tail from
   concern #8, and any future version of the same mistake.

If a cell below this one ever changes the splits, re-run this cell.

In [5]:
# --- Guardrail checks: each one fails loudly on a specific leak -------------------

# 1 · No dropped column made it into the feature list.
leaked = set(DROP_FROM_FEATURES) & set(FEATURES)
assert not leaked, f"leaky columns in FEATURES: {leaked}"

# 2 · Train and test never overlap in time.
assert train["REPDTE"].max() < test["REPDTE"].min(), "train and test windows overlap"

# 3 · Every training label resolves before the test window opens.
#     A row at quarter t is labeled from t+1..t+4, so push the last training date
#     forward by 4 quarters and check it still lands before the test starts.
last_train_label_resolves = train["REPDTE"].max() + pd.DateOffset(months=12)
assert last_train_label_resolves < test["REPDTE"].min(), (
    f"training labels resolve at {last_train_label_resolves.date()}, "
    f"inside the test window starting {test['REPDTE'].min().date()}"
)

# 4 · Both halves keep the true rare-event rate: no rebalancing has happened yet.
#     Lower bound is 0.0005, not 0.001: once the critical tier moved to Tier 1 tangible
#     equity the genuine test base rate fell to ~0.10%. The old 0.001 floor was set
#     against the EQV label and would now fail on a correct panel.
for name, df in [("train", train), ("test", test)]:
    rate = df[TARGET].mean()
    assert 0.0005 < rate < 0.05, f"{name} positive rate {rate:.4f} is not the real base rate"

# 5 · Every test quarter has real negatives (guards the positives-only tail, concern #8).
quarters_without_negatives = [
    d.date() for d, s in test.groupby("REPDTE")[TARGET] if not (s == 0).any()
]
assert not quarters_without_negatives, (
    f"test quarters with no negatives: {quarters_without_negatives}"
)

print("all leakage checks passed")
print(f"  train: {train['REPDTE'].min().date()} to {train['REPDTE'].max().date()}")
print(f"  test:  {test['REPDTE'].min().date()} to {test['REPDTE'].max().date()}")
print(f"  gap:   {gap['REPDTE'].min().date()} to {gap['REPDTE'].max().date()} (excluded)")

all leakage checks passed
  train: 1990-03-31 to 2015-12-31
  test:  2017-03-31 to 2025-03-31
  gap:   2016-03-31 to 2016-12-31 (excluded)


### What this does not cover

Three leakage risks live in code that doesn't exist yet. They get handled where they arise,
not here:

- **Trend features.** The funding signals still to build (deposit growth, uninsured-deposit
  share, held-to-maturity losses) must use backward-only windows. A rolling window centred on
  the prediction quarter, or any `shift(-1)`, reads the future.
- **Feature ranking.** The single-feature scores (AUC) in `eda.ipynb` were computed across
  all years, test period included. Re-rank on `train` only, then freeze the list.
- **Filling blanks and scaling.** Fit on `train`, then apply to `test`, never a statistic
  computed over both. Tree models (gradient boosting) take blanks natively and sidestep this.

One remaining limitation, not a fix: the economic columns (`FRED`) are treated as available
in their own quarter, but several publish late, GDP (`GDPC1`) about a month after the
quarter closes, house prices (`USSTHPI`) and lending standards (`DRTSCILM`) later still, all
revised afterward. Milder than `USREC`, since the lag is short and fixed rather than a
committee decision. Lag the macro block one quarter to be strict, or note it in the writeup.

## What is being compared: three benchmarks, four methods

A single comparison answers a single objection. Four reference points are used here because
four different objections get raised about a model like this.

**Benchmarks, nothing is fitted:**

| Benchmark | What it is | Objection it answers |
|---|---|---|
| **Naive floor** | Flag every bank | Is the model doing anything at all? |
| **Capital headroom** | Rank by how close the weakest capital ratio sits to its regulatory minimum | Does it beat what supervision already does? |
| **SCOR** | The FDIC's own off-site system, from the published record | Does it beat the system actually in production? |

**Methods, all fitted on the same training window and the same features:**

| Method | Role |
|---|---|
| **Gradient boosting** | Champion, the method being defended |
| **Logistic regression** | The alternative: simpler, explainable, and the thing the champion must beat |
| **MLP** | Neural network on the same tabular features (`deep_learning.ipynb`) |
| **GRU** | Sequence model reading eight quarters in order (`deep_learning.ipynb`) |

The naive floor exists because Correia/Luck/Verner (2024) report performance as a *multiple of*
what flagging everything achieves; a model that cannot beat it has found nothing. Capital
headroom is the operational status quo, Prompt Corrective Action *is* threshold monitoring.
SCOR is the only external number in the set that was measured on a comparable task.

By way of contrast, Carmona et al. (2018), the one paper here using this project's champion
method, reports no baseline at all. Their three "models" are three tuning stages of the same
XGBoost.

In [6]:
# The benchmark score is derived from the capital ratios, which are already features.
# Leaving it in would hand the models the benchmark's own answer.
BENCHMARK_COLS = ["headroom"]

# Text columns are held back for this first run so both models see identical inputs.
# One-hot encoding state (56 levels) would give gradient boosting a different feature
# space than logistic regression, confounding the comparison. Noted as a follow-up.
FEATURES = [
    c for c in panel.columns
    if c not in DROP_FROM_FEATURES + BENCHMARK_COLS
    and str(panel[c].dtype) not in ("object", "category")
]

print(f"model features: {len(FEATURES)}")
print(f"  of which trend: {sum(1 for c in FEATURES if '_chg' in c or '_grow' in c)}")
print(f"held back (text, for later): {CATEGORICAL}")

model features: 109
  of which trend: 50
held back (text, for later): ['STALP', 'BKCLASS', 'REGAGNT']


## How performance is measured

Accuracy is useless here. Predicting "no distress" for every bank scores **99.90%** and catches
nothing. None of the eleven papers reviewed uses it.

Three numbers instead, none of which depends on how many banks you decide to flag:

| Metric | Question it answers |
|---|---|
| **ROC-AUC** | Given one distressed and one healthy bank, how often is the distressed one ranked higher? 0.5 is a coin flip |
| **PR-AUC** | Of the banks flagged, what share are genuinely distressed, averaged over every possible cutoff? Sensitive to rarity, unlike ROC-AUC |
| **Lift** | PR-AUC divided by the base rate: how many times better than flagging at random. Correia/Luck/Verner's presentation |

Operational performance, how many cases you catch if you can examine *N* banks, is a separate
question, and it is reported in the alert-budget ladder below rather than folded in here. That
keeps the choice of budget visible instead of buried in a column header.

In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score


def evaluate(name, scores, y):
    """Score one ranking. Higher `scores` must mean higher risk.

    Every metric here is threshold-free -- none depends on how many banks get
    flagged. Anything that does belongs in the alert-budget ladder further down.
    """
    scores = np.asarray(scores, dtype=float)
    pr_auc = average_precision_score(y, scores)
    return {
        "model": name,
        "ROC-AUC": roc_auc_score(y, scores),
        "PR-AUC": pr_auc,
        "lift": pr_auc / y.mean(),        # multiple of what flagging at random achieves
    }

## Fitting the two models

**Logistic regression** cannot handle blanks or wildly different scales, so it gets a
pipeline: fill blanks with the median, then standardise. Both steps are fitted on the
**training data only** and then applied to test, fitting them on everything would leak
test-period information into training, the preprocessing leak flagged at the end of this
notebook.

**Gradient boosting** takes blanks natively and is scale-invariant, so it gets the raw
columns. That is one of the reasons it suits this data, where the era gaps are structural.

Neither model is tuned beyond sensible defaults. Tuning until the champion wins is how a
comparison becomes an argument for a foregone conclusion.

In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_train, y_train = train[FEATURES], train[TARGET].values
X_test,  y_test  = test[FEATURES],  test[TARGET].values

# Columns with no observed value at all in the training window cannot be imputed;
# drop them for the linear model only. (CET1 is empty before 2015.)
LR_FEATURES = [c for c in FEATURES if X_train[c].notna().any()]

logistic = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000),
).fit(X_train[LR_FEATURES], y_train)

# class_weight="balanced" tells the model a missed case costs as much as the class is
# rare -- the rebalancing Petropoulos et al. (2020) achieve by downsampling instead.
# Applied to the training fit only; the test set keeps its true 0.10% rate.
boosting = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.05,
    min_samples_leaf=50,
    l2_regularization=1.0,
    class_weight="balanced",
    random_state=0,
).fit(X_train, y_train)

print(f"logistic regression: {len(LR_FEATURES)} features")
print(f"gradient boosting:   {len(FEATURES)} features")

logistic regression: 109 features
gradient boosting:   109 features


## Scoring rows whose outcome was never observed

The diagnostic above settles whether the merged-away banks contaminate the *label*. A separate
question is whether they should be *scored* at all.

If the model flags a bank and that bank is acquired before its four-quarter window closes,
nobody ever learns whether the warning was right. Counted as it stands, that is a false alarmthe model is charged for an error that was never demonstrated. At a 1% alert budget, 125 of the
1,551 false positives are exactly this case, appearing at roughly 2.4 times their share of the
test set.

This is **censoring**, and survival analysis has a settled answer: an observation whose outcome
was not observed is excluded from scoring, neither a hit nor a miss. That is the treatment
adopted here, with the full test set reported underneath as a sensitivity check.

Note this is an evaluation decision, never a modelling one. At prediction time nobody knows a
bank is about to be acquired, so nothing about the merger can enter the features, that is
precisely the `near_merge_exit` leak the guardrail exists to block.

In [9]:
# Censored: the outcome window never closed because the bank left the panel by merger.
test_scored = test[~test["near_merge_exit"]]
X_test_scored = test_scored[FEATURES]
y_test_scored = test_scored[TARGET].values

print(f"test rows            {len(test):,}")
print(f"censored (excluded)  {test['near_merge_exit'].sum():,}")
print(f"scored               {len(test_scored):,}  "
      f"({int(y_test_scored.sum())} cases, base rate {y_test_scored.mean():.4f})")

test rows            166,754
censored (excluded)  5,637
scored               161,117  (160 cases, base rate 0.0010)


### Effect of the exclusion

It lifts PR-AUC by roughly 8%, and lifts the benchmark by the same proportion, because the
removed rows are hard negatives for every ranking, not just the model's. The miss rate does not
move at all, since every censored row is a negative and none was ever a case that could be
caught.

So the decision is about correctness, not advantage. Both versions appear in the results below.

## Results: test period 2017Q1 to 2025Q1

Primary figures use the scored set, with the censored rows removed as decided above. The full
test set is reported underneath so the effect of that decision is visible rather than assumed.

In [10]:
headroom_score = -test["headroom"].fillna(test["headroom"].max()).values
headroom_scored = -test_scored["headroom"].fillna(test_scored["headroom"].max()).values

results = pd.DataFrame([
    evaluate("naive (flag everything)",
             np.random.RandomState(0).rand(len(y_test_scored)), y_test_scored),
    evaluate("capital headroom", headroom_scored, y_test_scored),
    evaluate("logistic regression",
             logistic.predict_proba(X_test_scored[LR_FEATURES])[:, 1], y_test_scored),
    evaluate("gradient boosting",
             boosting.predict_proba(X_test_scored)[:, 1], y_test_scored),
])

print(f"scored rows {len(test_scored):,} | cases {int(y_test_scored.sum()):,} "
      f"| base rate {y_test_scored.mean():.4f}\n")
print(results.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

sensitivity = pd.DataFrame([
    evaluate("capital headroom", headroom_score, y_test),
    evaluate("logistic regression",
             logistic.predict_proba(X_test[LR_FEATURES])[:, 1], y_test),
    evaluate("gradient boosting", boosting.predict_proba(X_test)[:, 1], y_test),
])
print(f"\n\nSENSITIVITY, all {len(test):,} test rows, censored counted as false alarms\n")
print(sensitivity.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

scored rows 161,117 | cases 160 | base rate 0.0010

                  model  ROC-AUC  PR-AUC     lift
naive (flag everything)   0.5378  0.0012   1.1642
       capital headroom   0.8682  0.1282 129.1116
    logistic regression   0.9001  0.0307  30.8945
      gradient boosting   0.9190  0.2194 220.9655




SENSITIVITY, all 166,754 test rows, censored counted as false alarms

              model  ROC-AUC  PR-AUC     lift
   capital headroom   0.8677  0.1185 123.5436
logistic regression   0.9001  0.0285  29.6863
  gradient boosting   0.9189  0.2033 211.8311


## Diagnostic: does it hold when the failure mechanism changes?

The training years (1990–2015) are dominated by credit-driven distress: bad loans, real
estate, slow deterioration. The 2017–2025 test period contains the 2021–23 rate shock, a
different mechanism. A model that had merely memorised the credit-crisis pattern would win
on one era and lose on the other.

So: train through 2005, skip 2006, and test on the 2008 crisis, a period the model has
never seen, but which fails the way its training years did.

In [11]:
def run_split(train_end, test_start, test_end, label):
    """Refit everything on a different time split and return the comparison table."""
    tr = labeled[labeled["REPDTE"] <= train_end]
    te = labeled[(labeled["REPDTE"] >= test_start) & (labeled["REPDTE"] <= test_end)]
    y = te[TARGET].values

    lr_cols = [c for c in FEATURES if tr[c].notna().any()]
    lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                       LogisticRegression(max_iter=1000)).fit(tr[lr_cols], tr[TARGET])
    gb = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
                                        min_samples_leaf=50, l2_regularization=1.0,
                                        random_state=0).fit(tr[FEATURES], tr[TARGET])

    out = pd.DataFrame([
        evaluate("capital headroom", -te["headroom"].fillna(te["headroom"].max()).values, y),
        evaluate("logistic regression", lr.predict_proba(te[lr_cols])[:, 1], y),
        evaluate("gradient boosting", gb.predict_proba(te[FEATURES])[:, 1], y),
    ])
    print(f"\n{label}  |  train <= {train_end} ({len(tr):,})  ->  "
          f"test {test_start}..{test_end} ({len(te):,}, base {y.mean():.4f})")
    print(out.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    return out


crisis_2008 = run_split("2005-12-31", "2007-01-01", "2011-12-31", "2008 CREDIT CRISIS")


2008 CREDIT CRISIS  |  train <= 2005-12-31 (722,970)  ->  test 2007-01-01..2011-12-31 (160,590, base 0.0191)
              model  ROC-AUC  PR-AUC    lift
   capital headroom   0.8817  0.2976 15.5561
logistic regression   0.9280  0.3102 16.2167
  gradient boosting   0.9452  0.4214 22.0309


## Reading the two tables together

Same model, same features, same code, two eras, and the champion beats the benchmark in
both. The margin is larger where the mechanism matches the training data (PR-AUC 0.421
against 0.298 on 2008) than on the mixed modern period (0.219 against 0.128), which is the
direction you would expect, but there is no era where it loses.

One honest caveat carries over from Petropoulos et al. (2020): out-of-time transfer is the
harder test, and the one weak test year (2022, below) shows the mechanism shift has not
disappeared, it has shrunk to a single year.

## Diagnostics: locating where the model actually fails

The headline table is one number covering nine years, which hides more than it shows. Two
sweeps break it apart. Both are **diagnostics, not model selection**: the reported model
above is fixed, and nothing below is used to choose it. That distinction matters, because
running many variants and reporting the winner would quietly turn the test set into a
training set.

### Sweep 1: does older training data help or hurt?

The training window runs from 1990, but the 1990s and the 2000s were different banking
industries under different capital rules. If the early years are teaching the wrong lesson,
cutting them should improve performance on a modern test period.

Nine training start years, one fixed test period.

In [12]:
def train_from(start_year):
    """Refit the champion on a shortened training window. Test period never changes."""
    tr = train[train["REPDTE"] >= f"{start_year}-01-01"]
    m = HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, min_samples_leaf=50,
        l2_regularization=1.0, class_weight="balanced", random_state=0,
    ).fit(tr[FEATURES], tr[TARGET])
    row = evaluate(str(start_year), m.predict_proba(X_test)[:, 1], y_test)
    row["train_rows"], row["train_pos"] = len(tr), int(tr[TARGET].sum())
    return row


sweep_era = pd.DataFrame([train_from(y) for y in
                          [1990, 1993, 1996, 1999, 2002, 2005, 2008, 2010, 2012]])
sweep_era = sweep_era.rename(columns={"model": "train from"})
print(sweep_era[["train from", "train_rows", "train_pos",
                 "ROC-AUC", "PR-AUC", "lift"]].to_string(
      index=False, float_format=lambda v: f"{v:.4f}"))

train from  train_rows  train_pos  ROC-AUC  PR-AUC     lift
      1990     1026797       8414   0.9189  0.2033 211.8311
      1993      860630       5174   0.9232  0.2059 214.5403
      1996      706901       4596   0.9173  0.2046 213.2546
      1999      572676       4116   0.9170  0.1954 203.6980
      2002      451527       3734   0.9081  0.1978 206.1085
      2005      339634       3556   0.8977  0.2169 226.0893
      2008      233884       3154   0.8989  0.2071 215.8900
      2010      168015       1231   0.9103  0.2293 238.9481
      2012      107926        370   0.8582  0.1045 108.9303


### Reading Sweep 1: the era does not matter much, and 2010 is a mirage

Everything from 1993 to 2005 lands in a narrow band around 0.20 PR-AUC. There is no cliff,
so there is no regime break hiding in the training data.

The one apparent standout, 2010, is worth looking at closely, because it is exactly the kind
of result that gets reported without scrutiny:

| Train from | PR-AUC |
|---|---|
| 2008 | 0.207 |
| **2010** | **0.229** |
| 2012 | 0.105 |

A genuine effect would trend. A single high point wedged between two low neighbours is
sampling noise, and by 2012 the training set holds only 370 positives, so the estimates
there are unstable to begin with. Cutting to 2010 would also discard the 2008 crisis, the
densest source of real distress cases in the data.

**Decision: keep the full 1990–2015 window.** Not because it scored best, but because the
sweep shows nothing is gained by cutting, and cutting invites a choice made on test results.

### Sweep 2: when does the model stop working?

The same fixed model, scored one test year at a time. If performance degrades smoothly, the
problem is drift. If it falls off a cliff in specific years, something changed in those years.

Base rates differ year to year, so PR-AUC is also shown as **lift**, the multiple of what
flagging at random would achieve, which is comparable across rows.

In [13]:
scored = test.copy()
scored["model_score"] = boosting.predict_proba(X_test)[:, 1]
scored["bench_score"] = -scored["headroom"].fillna(scored["headroom"].max())

rows = []
for year, block in scored.groupby(scored["REPDTE"].dt.year):
    y = block[TARGET].values
    if y.sum() < 3:                      # too few cases for a stable estimate
        continue
    model_pr = average_precision_score(y, block["model_score"])
    bench_pr = average_precision_score(y, block["bench_score"])
    rows.append({
        "year": year, "banks": len(block), "cases": int(y.sum()),
        "base rate": y.mean(),
        "ROC-AUC": roc_auc_score(y, block["model_score"]),
        "model lift": model_pr / y.mean(),
        "bench lift": bench_pr / y.mean(),
    })

sweep_year = pd.DataFrame(rows)
print(sweep_year.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

hard = sweep_year[sweep_year["year"].between(2021, 2023)]["cases"].sum()
print(f"\n2021-2023 holds {hard} of {int(y_test.sum())} test cases "
      f"({hard / y_test.sum() * 100:.0f}%)")

 year  banks  cases  base rate  ROC-AUC  model lift  bench lift
 2017  23189     24     0.0010   0.9451    198.3837    197.5401
 2018  22188     37     0.0017   0.9515    214.5959    112.2534
 2019  21253     17     0.0008   0.9660    609.6428    264.8306
 2020  20363      5     0.0002   0.9618     31.8607      2.8716
 2021  19561     17     0.0009   0.9426    350.9518    312.3156
 2022  18893     17     0.0009   0.7161    111.2033    105.1534
 2023  18613     13     0.0007   0.9695    229.2859     71.8481
 2024  18203     24     0.0013   0.9769    295.5312    136.3053
 2025   4491      6     0.0013   0.9211    301.6725    366.1000

2021-2023 holds 47 of 160 test cases (29%)


### Reading Sweep 2: one weak year, not a collapse

| Period | ROC-AUC | Verdict |
|---|---|---|
| 2017–2021 | 0.94–0.97 | Strong, every year |
| **2022** | **0.72** | The one weak year (17 cases) |
| 2023–2025 | 0.92–0.98 | Strong again |

The cases are also no longer concentrated in the weak window: 2021–2023 holds 47 of the
160 test cases, 29%. The single weak year is 2022, when the rate shock was moving fastestthe quarter-over-quarter repricing the annual training patterns have least to say about.
It holds 17 cases, so the estimate is itself noisy.

**What this changes about the claim.** The champion beats the status quo on average *and*
in eight of nine individual years. The remaining honest qualifier is 2022, and it belongs
in the writeup as a limitation, not a collapse.

## Do the macro columns earn their place?

The panel joins eleven FRED series, rates, unemployment, GDP, house prices, credit spreadsfinancial stress. That join was made on the assumption it was literature-backed. Re-reading
the source showed the opposite: Nuxoll (2003) ran exactly this test and concluded that
*"economic data do not improve these forecasts despite the fact that the data are
statistically significant."* Oshinsky & Olin (2005) excluded economic variables by choice.

So the join needs its own evidence. The test is an ablation, the same model, fitted twicewith and without the macro block, which is the standard move in this literature (Nuxoll runs
it for economic data, Curry et al. for market data).

Run on both test periods, since a variable that is useless in calm years might still matter
in a crisis.

In [14]:
MACRO = ["FEDFUNDS", "DGS10", "T10Y3M", "UNRATE", "GDPC1", "CPIAUCSL",
         "USSTHPI", "BAA10Y", "DRTSCILM", "NFCI", "BOGZ1FL075035503Q"]
WITHOUT_MACRO = [c for c in FEATURES if c not in MACRO]


def ablate(features, tr, te, label):
    m = HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, min_samples_leaf=50,
        l2_regularization=1.0, class_weight="balanced", random_state=0,
    ).fit(tr[features], tr[TARGET])
    return evaluate(label, m.predict_proba(te[features])[:, 1], te[TARGET].values)


train_2005 = labeled[labeled["REPDTE"] <= "2005-12-31"]
test_2008 = labeled[(labeled["REPDTE"] >= "2007-01-01") & (labeled["REPDTE"] <= "2011-12-31")]

ablation = pd.DataFrame([
    ablate(FEATURES,      train,      test,      "2017-2025  with macro"),
    ablate(WITHOUT_MACRO, train,      test,      "2017-2025  without macro"),
    ablate(FEATURES,      train_2005, test_2008, "2008 crisis  with macro"),
    ablate(WITHOUT_MACRO, train_2005, test_2008, "2008 crisis  without macro"),
])

print(f"{len(FEATURES)} features with macro, {len(WITHOUT_MACRO)} without\n")
print(ablation.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

109 features with macro, 98 without

                     model  ROC-AUC  PR-AUC     lift
     2017-2025  with macro   0.9189  0.2033 211.8311
  2017-2025  without macro   0.9133  0.2185 227.6904
   2008 crisis  with macro   0.9479  0.4593  24.0092
2008 crisis  without macro   0.9399  0.4269  22.3189


### Verdict: Nuxoll was right, and still is

On the main test period the macro block does not help, without it, PR-AUC is actually
slightly *higher* (0.219 against 0.203). On the 2008 crisis it adds about 8% of PR-AUC
(0.459 against 0.427). Small, and consistent with the idea that aggregate conditions
matter most when the whole system is moving together.

**Why this happens.** The macro columns hold the same eleven numbers for every bank in a
given quarter. They can raise or lower the predicted risk of the entire population at once,
but they carry no information about *which* bank is the problem, and picking which bank is
the entire task. That is Nuxoll's own explanation, and it holds on data running twenty-three
years past his.

**Decision: keep the block, and report this.** The ablation is worth more in the writeup
than the features are in the model, an assumption inherited from a misread citation, tested
directly, and resolved against a published result.

The related caveat from the guardrail section still stands: several of these series publish
weeks after the quarter they describe, so a strict version would lag the whole block by one
quarter. Given they contribute nothing measurable, that refinement is not worth the
complexity.

## Is the label contaminated by banks that merged away?

Concern #3 in `data_concerns.md`: when a bank's charter ends by merger, it leaves the panel,
so its last four quarters were labelled "stayed safe" without anyone observing whether it
would have. Those rows carry the `near_merge_exit` flag, 48,239 of them, about 4% of the
healthy negatives.

The worry is not academic. A weak bank often escapes by being acquired, and the Week 1
proposal lists voluntary merger as one of the corrective actions available to a distressed
bank. If that is common, those rows are distress cases sitting in the data labelled safe, and
every score above is understated.

Oshinsky & Olin (2005) handle this by making merger its own outcome in a four-way model. That
is a redesign. The cheaper first question is whether there is anything to redesign *for*: do
these banks look like the distressed ones, or like the survivors?

In [15]:
VITALS = ["RBCRWAJ", "NPERFV", "ROA", "ROE", "EEFFR", "ORER"]
TRENDS = ["RBCRWAJ_chg8q", "NPERFV_chg8q", "ROA_chg8q", "EQV_chg8q"]

negatives = labeled[labeled[TARGET] == 0]
merged     = negatives[negatives["near_merge_exit"]]
survivors  = negatives[~negatives["near_merge_exit"]]
distressed = labeled[labeled[TARGET] == 1]

groups = {"merged away": merged, "healthy survivors": survivors,
          "became undercapitalized": distressed}

print("LEVELS, median vitals\n")
print(pd.DataFrame({k: v[VITALS].median() for k, v in groups.items()}).round(2).to_string())

print("\n\nTRENDS, median change over 8 quarters\n")
print(pd.DataFrame({k: v[TRENDS].median() for k, v in groups.items()}).round(3).to_string())

# What share of each group would a crude weakness screen pick up?
weak_capital = survivors["RBCRWAJ"].quantile(0.10)
weak_npa = survivors["NPERFV"].quantile(0.90)
print(f"\n\nSHARE LOOKING WEAK  (capital < {weak_capital:.2f}% or bad loans > {weak_npa:.2f}%)\n")
for name, block in groups.items():
    share = ((block["RBCRWAJ"] < weak_capital) | (block["NPERFV"] > weak_npa)).mean()
    print(f"  {name:<26} {share:6.1%}   (n = {len(block):,})")

print("\n\nSENSITIVITY, does removing them move the target rate?\n")
print(f"  all rows          {labeled[TARGET].mean():.3%}")
print(f"  merged excluded   {labeled[~labeled['near_merge_exit']][TARGET].mean():.3%}")

LEVELS, median vitals

         merged away  healthy survivors  became undercapitalized
RBCRWAJ        14.61              15.77                    10.10
NPERFV          0.59               0.57                     4.35
ROA             0.91               1.01                    -0.36
ROE             9.34              10.09                    -5.03
EEFFR          66.76              65.80                    88.94
ORER            0.03               0.03                     0.86


TRENDS, median change over 8 quarters

               merged away  healthy survivors  became undercapitalized
RBCRWAJ_chg8q        0.212              0.115                   -1.611
NPERFV_chg8q        -0.056             -0.024                    2.866
ROA_chg8q           -0.002              0.021                   -0.981
EQV_chg8q            0.259              0.156                   -1.320


SHARE LOOKING WEAK  (capital < 11.22% or bad loans > 2.62%)

  merged away                 21.9%   (n = 48,262)
  healthy su

  merged excluded   0.737%


### Verdict: normal consolidation, and the label stands

The merged-away banks track the survivors on every measure, and are nowhere near the
distressed profile:

| Median | Merged away | Survivors | Became undercapitalized |
|---|---|---|---|
| Capital ratio | 14.61 | 15.77 | **10.10** |
| Bad loans | 0.59 | 0.57 | **4.35** |
| ROA | 0.91 | 1.01 | **−0.36** |
| Capital, 2-yr change | **+0.21** | +0.12 | **−1.61** |

The trend row is the decisive one. Banks heading for distress lose 1.61 points of capital
over two years, the decline the EDA chart is built on. Banks that merged away were *gaining*
capital, slightly faster than the survivors. Whatever drove those mergers, it was not the
deterioration this project is trying to catch.

The weakness screen agrees: 21.9% of merged banks trip it, against 18.1% of survivors and
85.7% of genuine cases. There is a mild tilt, some weak banks clearly are bought, but it is
a few points, not a hidden population.

**Decision: keep the rows, keep the label, drop the concern.** Removing all 48,262 shifts the
target rate from 0.708% to 0.737%, which changes nothing that matters. The four-way
multinomial redesign is not warranted: it would spend the project's remaining time modelling
an outcome that turns out to be ordinary industry consolidation, 69% of the panel by count.

This closes concern #3. The `near_merge_exit` flag stays in the table as the evidence for the
check, not as a correction to be applied.

---

# Deep learning challengers

Two neural networks on the same split, the same censored-row exclusion, and the same
metrics as everything above.

| Model | What it sees |
|---|---|
| **MLP** (multi-layer perceptron) | One quarter per bank, the same features gradient boosting gets |
| **GRU** (gated recurrent unit) | Eight quarters in order, so it reads the *shape* of a decline rather than being handed "capital fell 2 points" |

The GRU is the interesting one. This project's central EDA finding is that banks decline
over two years before crossing into distress, and a sequence model is the natural way to
learn that shape. Whether it beats features that hand the trend over directly is an open
question, and the answer below is no.

Petropoulos et al. (2020) tested neural networks on this problem and found them second to
tree ensembles, so there is precedent either way.

## Setup

One structural difference from the models above: early stopping needs a validation set, and it
must not be the test period. It is carved from the **end of training**, 2011–2015, which holds
about 670 cases. Stopping on 2013+ alone was tried first and abandoned; 220 cases gives a signal
too noisy to stop on.

So this section uses train ≤ 2010, validate 2011–2015, and the same test period as everything
else. Its variables are prefixed `dl_` so the splits used earlier stay untouched.

> torch and LightGBM each bundle their own OpenMP runtime, and importing both in one process
> aborts. The guard below must run before torch loads.

In [16]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
import torch.nn as nn

DEV = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {DEV}")

DL_FEATURES = FEATURES                       # identical inputs to the models above

dl_all = labeled[labeled["REPDTE"] <= TRAIN_END]
dl_train = dl_all[dl_all["REPDTE"] <= "2010-12-31"]      # fit here
dl_val = dl_all[dl_all["REPDTE"] >= "2011-01-01"]        # early stopping here
dl_test = test_scored                                     # same scored set as above

print(f"train {len(dl_train):,} ({int(dl_train[TARGET].sum())} cases) | "
      f"val {len(dl_val):,} ({int(dl_val[TARGET].sum())}) | "
      f"test {len(dl_test):,} ({int(dl_test[TARGET].sum())})")

device: mps


train 889,445 (7748 cases) | val 137,352 (666) | test 161,117 (160)


## Training loop

Nothing exotic: class-weighted loss so the rare cases are not drowned out, AdamW, and early
stopping on validation PR-AUC with a patience of five. The best weights by validation score are
restored before predicting, so a late epoch that overfits cannot be the one reported.

In [17]:
def train_net(model, Xtr, ytr, Xva, yva, epochs=40, bs=4096, lr=1e-3, patience=5):
    """Fit one network. Returns it with the best-validation weights restored."""
    model = model.to(DEV)
    ratio = float((ytr == 0).sum()) / max(float((ytr == 1).sum()), 1.0)
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([ratio], dtype=torch.float32, device=DEV))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    Xva_t = torch.tensor(Xva, dtype=torch.float32, device=DEV)
    best_score, best_weights, since_improved = -1.0, None, 0

    for epoch in range(epochs):
        model.train()
        # Rows arrive sorted by bank, so consecutive rows are the same bank in
        # adjacent quarters. Without shuffling, every batch is a handful of banks
        # and the gradient is badly correlated.
        order = np.random.permutation(len(Xtr))
        for i in range(0, len(Xtr), bs):
            idx = order[i:i + bs]
            xb = torch.tensor(Xtr[idx], dtype=torch.float32, device=DEV)
            yb = torch.tensor(ytr[idx], dtype=torch.float32, device=DEV)
            opt.zero_grad()
            loss_fn(model(xb).squeeze(-1), yb).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            preds = torch.sigmoid(model(Xva_t).squeeze(-1)).cpu().numpy()
        score = average_precision_score(yva, preds)

        if score > best_score:
            best_score = score
            best_weights = {k: v.detach().clone() for k, v in model.state_dict().items()}
            since_improved = 0
        else:
            since_improved += 1
            if since_improved >= patience:
                break

    model.load_state_dict(best_weights)
    print(f"    best validation PR-AUC {best_score:.4f}")
    return model


def net_predict(model, X, bs=16384):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb = torch.tensor(X[i:i + bs], dtype=torch.float32, device=DEV)
            out.append(torch.sigmoid(model(xb).squeeze(-1)).cpu().numpy())
    return np.concatenate(out)


dl_results = []
dl_scores = {}          # kept so the summary can score at other budgets

## 1 · MLP on the tabular features

Two hidden layers with dropout, on the same features gradient boosting sees.

**One trap, found the hard way.** Filling blanks with the training median leaves blanks in place
for any column that is *entirely* blank in training, and blanks propagate through a networkturning every prediction into `NaN`. The model then scores exactly 0.5 ROC-AUC and 0% recall,
which reads like a failed method rather than a broken pipeline. CET1 (`RBCT1CER`) is empty before
2015 and does exactly this. The assertion below makes it impossible to miss again.

In [18]:
MLP_FEATURES = [c for c in DL_FEATURES if dl_train[c].notna().any()]
print(f"features: {len(MLP_FEATURES)} "
      f"(dropped as all-blank in training: {sorted(set(DL_FEATURES) - set(MLP_FEATURES))})")

fill = dl_train[MLP_FEATURES].median()


def as_matrix(df):
    return df[MLP_FEATURES].fillna(fill).to_numpy(np.float32)


Xtr = as_matrix(dl_train)
centre, spread = Xtr.mean(0), Xtr.std(0) + 1e-6
scale = lambda X: np.clip((X - centre) / spread, -10, 10)

Xtr = scale(Xtr)
Xva = scale(as_matrix(dl_val))
Xte = scale(as_matrix(dl_test))
assert not np.isnan(Xtr).any(), "blanks survived into the training matrix"

ytr = dl_train[TARGET].values.astype(np.float32)
yva = dl_val[TARGET].values.astype(np.float32)
yte = dl_test[TARGET].values.astype(np.float32)

SEEDS = [0, 1, 2]
for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    mlp = nn.Sequential(
        nn.Linear(len(MLP_FEATURES), 256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(64, 1),
    )
    mlp = train_net(mlp, Xtr, ytr, Xva, yva)
    scores = net_predict(mlp, Xte)
    dl_scores.setdefault("MLP", []).append(scores)
    dl_results.append(evaluate(f"MLP (seed {seed})", scores, yte))

features: 108 (dropped as all-blank in training: ['RBCT1CER'])


    best validation PR-AUC 0.3848


    best validation PR-AUC 0.3418


    best validation PR-AUC 0.3669


## 2 · GRU on eight-quarter sequences

Each row becomes a sequence: the bank's last eight quarters of twenty core measures, oldest to
newest. The lookback is built with the same self-join on `(bank, quarter − h)` used for the trend
features, so a gap in a bank's history yields a blank rather than the wrong quarter.

Two details that matter:

- **Mask channels.** Blanks are filled with the training median, and a parallel set of 0/1
  channels tells the network which values were filled. Without this it cannot distinguish a bank
  sitting at the median from a bank with no filing at all.
- **Downsampled negatives.** 300,000 negatives plus every case, the approach Petropoulos et al.
  take. Validation and test keep their true distribution, only the training fold is touched.

In [19]:
SEQ_FEATS = ["RBCRWAJ", "RBC1AAJ", "EQV", "NPERFV", "NCLNLSR", "ORER", "NTLNLSR",
             "LNATRESR", "ROA", "ROE", "NIMY", "EEFFR", "LNLSDEPR", "CHBALR",
             "BROR", "uninsured_pct", "htm_loss_pct", "INTINCY", "INTEXPY", "ASSET"]
LOOKBACK = 8

panel["_qi"] = panel["REPDTE"].dt.year * 4 + panel["REPDTE"].dt.quarter
history = panel[["CERT", "_qi"]].copy()
for h in range(LOOKBACK):
    past = panel[["CERT", "_qi"] + SEQ_FEATS].copy()
    past["_qi"] = past["_qi"] + h
    past = past.rename(columns={c: f"{c}_t{h}" for c in SEQ_FEATS})
    history = history.merge(past, on=["CERT", "_qi"], how="left")
history.index = panel.index
panel = panel.drop(columns=["_qi"])

SEQ_COLS = [f"{c}_t{h}" for h in range(LOOKBACK) for c in SEQ_FEATS]

# ASSET spans six orders of magnitude -- log it before scaling.
for h in range(LOOKBACK):
    history[f"ASSET_t{h}"] = np.log1p(history[f"ASSET_t{h}"].clip(lower=0))

rng = np.random.RandomState(0)
case_rows = dl_train.index[dl_train[TARGET] == 1]
other_rows = rng.choice(dl_train.index[dl_train[TARGET] == 0], size=300_000, replace=False)
seq_train_idx = np.concatenate([case_rows.values, other_rows])
rng.shuffle(seq_train_idx)
print(f"training sequences: {len(seq_train_idx):,} ({len(case_rows):,} cases)")

seq_fill = history.loc[seq_train_idx, SEQ_COLS].median()


def as_sequences(idx, centre=None, spread=None):
    block = history.loc[idx, SEQ_COLS]
    missing = block.isna().to_numpy(np.float32)
    values = block.fillna(seq_fill).to_numpy(np.float32)
    if centre is not None:
        values = np.clip((values - centre) / spread, -10, 10)
    n, F, L = len(values), len(SEQ_FEATS), LOOKBACK
    # Columns run lag-major; reshape then flip so the sequence reads oldest -> newest.
    values = values.reshape(n, L, F)[:, ::-1, :].copy()
    missing = missing.reshape(n, L, F)[:, ::-1, :].copy()
    return np.concatenate([values, missing], axis=2)


raw = history.loc[seq_train_idx, SEQ_COLS].fillna(seq_fill).to_numpy(np.float32)
seq_centre, seq_spread = raw.mean(0), raw.std(0) + 1e-6

Str = as_sequences(seq_train_idx, seq_centre, seq_spread)
Sva = as_sequences(dl_val.index, seq_centre, seq_spread)
Ste = as_sequences(dl_test.index, seq_centre, seq_spread)
ytr_seq = dl_train[TARGET].loc[seq_train_idx].values.astype(np.float32)
print(f"sequence tensor: {Str.shape}  (8 quarters x {len(SEQ_FEATS)} measures + mask)")


class GRUNet(nn.Module):
    def __init__(self, n_feat, hidden=64):
        super().__init__()
        self.gru = nn.GRU(n_feat, hidden, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(hidden, 1))

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1, :])       # last step = the prediction quarter


for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    gru = train_net(GRUNet(Str.shape[2]), Str, ytr_seq, Sva, yva,
                    epochs=30, bs=1024, lr=2e-3)
    scores = net_predict(gru, Ste)
    dl_scores.setdefault("GRU", []).append(scores)
    dl_results.append(evaluate(f"GRU (seed {seed})", scores, yte))

training sequences: 307,748 (7,748 cases)


sequence tensor: (307748, 8, 40)  (8 quarters x 20 measures + mask)


    best validation PR-AUC 0.3194


    best validation PR-AUC 0.3323


    best validation PR-AUC 0.3225


## Results

Each network is run on **three seeds**. This is not thoroughness for its own sake: an early run
of the GRU scored 0.062 and a rerun of identical code scored 0.051, purely from initialisation.
Where the spread between seeds rivals the spread between models, a single run is not a result.
Petropoulos bootstraps for the same reason.

In [20]:
deep = pd.DataFrame(dl_results)
print(deep.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

deep["family"] = deep["model"].str.replace(r" \(seed \d\)", "", regex=True)
print("\n\nacross seeds:\n")
print(deep.groupby("family").agg(
    runs=("PR-AUC", "size"),
    roc=("ROC-AUC", "mean"),
    pr_mean=("PR-AUC", "mean"),
    pr_min=("PR-AUC", "min"),
    pr_max=("PR-AUC", "max"),
    lift=("lift", "mean"),
).to_string(float_format=lambda v: f"{v:.4f}"))

       model  ROC-AUC  PR-AUC     lift
MLP (seed 0)   0.8816  0.1079 108.6267
MLP (seed 1)   0.8844  0.0617  62.1036
MLP (seed 2)   0.8833  0.0871  87.6999
GRU (seed 0)   0.9231  0.0880  88.5961
GRU (seed 1)   0.9082  0.1332 134.1096
GRU (seed 2)   0.8969  0.1438 144.8490


across seeds:

        runs    roc  pr_mean  pr_min  pr_max     lift
family                                               
GRU        3 0.9094   0.1217  0.0880  0.1438 122.5182
MLP        3 0.8831   0.0855  0.0617  0.1079  86.1434


### Verdict

**Neither network earns the champion slot.** The GRU is the better of the two, ROC-AUC
0.909 and mean PR-AUC 0.122, just below the untrained benchmark's 0.128, and the MLP sits
below both (0.883 / 0.086). Both are well behind gradient boosting on every metric.

The sequence model's near-miss is still the more interesting result. It was the natural fit
for a two-year-decline story, and it roughly matches a benchmark that reads one ratio, but
the 4- and 8-quarter change columns built in `feature_engineering.ipynb` appear to already
carry what the sequence carries, so the GRU pays the cost of learning the shape from scratch
with little left to gain.

**And the seed variance is the methodological point.** Across three seeds the MLP's PR-AUC
spans 0.062–0.108 and the GRU's 0.088–0.144, swings of roughly half the mean, comparable to
the gaps between different models. Any single-run neural network number on this data should
be treated as one draw, not as a measurement.

---

# Operational view: the alert-budget ladder

Placed after every model has been fitted, so all six can be shown together.

## First, context: how well does the FDIC's own system do?

Every comparison above is against something built in this notebook. It is worth knowing what a
real off-site monitoring system achieves, not to compete with it, but to calibrate what
"good" looks like on this kind of problem.

**SCOR** is the FDIC's off-site system. It scores every bank's call report between examinations
and flags those likely to be downgraded at the next one. Collier et al. (2003) publish its
year-by-year record, 1986–2002:

| | SCOR, 1986–2002 |
|---|---|
| Bank-examinations | 126,505 |
| Actual downgrades | 7,162 (5.66% base rate) |
| Flagged | 8,622 (6.8% of those examined) |
| Correct | 3,196 |
| **Caught** | **44.6%** |
| **False alarms** | **62.9%** |

**This is not a benchmark for this project, and it is deliberately not in the results table.**
SCOR predicts a *CAMELS downgrade*, an examiner's judgment, applied to banks already scheduled
for examination, at a four-to-six-month horizon. This project predicts a *capital ratio crossing
a statutory threshold*, across all healthy banks, four quarters out. The events differ, the
populations differ, and the base rates differ by a factor of sixteen. Scoring them side by side
would be comparing two different problems.

What it does establish is the difficulty of the task. **The FDIC, using its own supervisory data
on a denser and more forgiving target, catches fewer than half the cases it is looking for.**
Any claim that a model built from public call reports should be catching most of them is not
supported by the operational record.

Two further details worth carrying into the writeup:

- **The widely-quoted "about two-thirds missed" is loose.** Over the full record it is 55.4%,
  though it does exceed two-thirds after 1993.
- **SCOR degraded the same way this project's model does.** Its miss rate rose from 27–52% in
  1986–1991 to 76–89% across 1993–2002, built on the banking crisis of the late 1980s, it lost
  most of its power once the failure mechanism changed. The regime sensitivity found in Sweep 2
  is a known property of call-report monitoring, visible in the FDIC's own production system.

In [21]:
# SCOR's published record (Collier et al. 2003, Table 4), for context only.
SCOR = {"examined": 126_505, "downgraded": 7_162, "flagged": 8_622, "correct": 3_196}
print(f"SCOR, 1986-2002: caught {SCOR['correct'] / SCOR['downgraded']:.1%} of downgrades "
      f"at a {SCOR['flagged'] / SCOR['examined']:.1%} alert rate\n")

# Any single alert budget is arbitrary -- it depends on examiner capacity, not on the
# model. Cole & White (2012) report error rates across a range for the same reason.
BUDGETS = (0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009,
           0.01, 0.015, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09,
           0.10, 0.11, 0.12, 0.13, 0.14, 0.15,
           0.20, 0.25, 0.30)

# Every benchmark and method, including random, so the floor is visible.
ladder_scores = {
    "random": np.random.RandomState(0).rand(len(y_test_scored)),
    "capital headroom": headroom_scored,
    "logistic regression": logistic.predict_proba(X_test_scored[LR_FEATURES])[:, 1],
    "gradient boosting": boosting.predict_proba(X_test_scored)[:, 1],
    "MLP": np.mean(dl_scores["MLP"], axis=0),      # averaged over the three seeds
    "GRU": np.mean(dl_scores["GRU"], axis=0),
}


def caught_at(scores, y, budget):
    k = int(len(scores) * budget)
    return y[np.argsort(-np.asarray(scores, float))[:k]].sum() / y.sum()


ladder = pd.DataFrame(
    {name: [caught_at(s, y_test_scored, b) for b in BUDGETS]
     for name, s in ladder_scores.items()},
    index=[f"{b:.1%}  ({int(len(y_test_scored) * b):,} filings)" for b in BUDGETS],
)
ladder.index.name = "alert budget"

print("SHARE OF TROUBLED BANK-QUARTERS CAUGHT\n")
print(ladder.to_string(float_format=lambda v: f"{v:.1%}"))

# The error view, at the 1% rung only -- the other budgets scale predictably.
k = int(len(y_test_scored) * 0.01)
errors = pd.DataFrame([
    {"model": n,
     "caught": caught_at(s, y_test_scored, 0.01),
     "missed (Type I)": 1 - caught_at(s, y_test_scored, 0.01),
     "false alarms (Type II)":
         1 - caught_at(s, y_test_scored, 0.01) * y_test_scored.sum() / k}
    for n, s in ladder_scores.items()
])
print(f"\n\nERROR BREAKDOWN AT 1% ({k:,} filings flagged)\n")
print(errors.to_string(index=False, float_format=lambda v: f"{v:.1%}"))

SCOR, 1986-2002: caught 44.6% of downgrades at a 6.8% alert rate



SHARE OF TROUBLED BANK-QUARTERS CAUGHT

                         random  capital headroom  logistic regression  gradient boosting   MLP   GRU
alert budget                                                                                         
0.1%  (161 filings)        0.0%             26.2%                 7.5%              31.9% 21.9% 23.1%
0.2%  (322 filings)        0.0%             34.4%                15.0%              43.1% 32.5% 33.8%
0.3%  (483 filings)        0.0%             41.2%                19.4%              50.0% 35.0% 39.4%
0.4%  (644 filings)        0.0%             45.0%                23.1%              54.4% 40.0% 43.1%
0.5%  (805 filings)        0.6%             46.9%                25.0%              59.4% 43.1% 45.6%
0.6%  (966 filings)        0.6%             47.5%                25.6%              61.3% 44.4% 49.4%
0.7%  (1,127 filings)      0.6%             47.5%                30.6%              63.7% 46.9% 51.9%
0.8%  (1,288 filings)      0.6%           

In [22]:
# How many distinct banks sit inside each method's worst 1%, and how many of the
# troubled banks that reaches. The ladder above counts bank-quarters; supervisors
# schedule examinations by bank, so the bank-level count is the operational one.
K1 = int(len(y_test_scored) * 0.01)
certs = test_scored["CERT"].values
troubled_banks = set(certs[np.asarray(y_test_scored).astype(bool)])

rows = []
for name, s in ladder_scores.items():
    top = np.argsort(-np.asarray(s, float))[:K1]
    flagged = set(certs[top])
    rows.append({"model": name,
                 "banks flagged": len(flagged),
                 "troubled banks caught": len(flagged & troubled_banks)})

banks_at_1pct = pd.DataFrame(rows).set_index("model")
banks_at_1pct["share of troubled banks"] = (
    banks_at_1pct["troubled banks caught"] / len(troubled_banks) * 100).round(1)

print(f"At the 1% budget: {K1:,} filings flagged, "
      f"{len(troubled_banks)} distinct troubled banks in the test period\n")
print(banks_at_1pct.to_string())

At the 1% budget: 1,611 filings flagged, 44 distinct troubled banks in the test period

                     banks flagged  troubled banks caught  share of troubled banks
model                                                                             
random                        1411                      8                     18.2
capital headroom               404                     34                     77.3
logistic regression            399                     27                     61.4
gradient boosting              455                     36                     81.8
MLP                            496                     31                     70.5
GRU                            387                     33                     75.0


In [23]:
# Of the troubled bank-quarters the model catches at the 1% budget, how far ahead of
# the capital breach did each one sit? onset_4q marks the four quarters before a
# breach, so the lead runs 1 to 4 quarters by construction -- the question is which
# of those four the model actually reaches.
gb_scores = boosting.predict_proba(X_test_scored)[:, 1]
in_top = np.zeros(len(gb_scores), bool)
in_top[np.argsort(-gb_scores)[:K1]] = True

lead = test_scored[["CERT", "REPDTE"]].copy()
lead["REPDTE"] = pd.to_datetime(lead["REPDTE"])
lead["y"] = np.asarray(y_test_scored)
lead["caught"] = in_top
lead = lead[lead["y"] == 1].sort_values(["CERT", "REPDTE"]).reset_index(drop=True)

# split each bank's positives into contiguous episodes, then count back from the
# last quarter of the episode (which sits one quarter before the breach itself)
qi = lead["REPDTE"].dt.to_period("Q").astype("int64")
new_run = (lead["CERT"] != lead["CERT"].shift()) | (qi - qi.shift() != 1)
lead["run"] = new_run.cumsum()
lead["lead_q"] = lead.groupby("run")["REPDTE"].transform("max").dt.to_period("Q").astype("int64") - qi + 1

tab = pd.DataFrame({"all cases": lead.groupby("lead_q").size(),
                    "caught": lead[lead["caught"]].groupby("lead_q").size()}).fillna(0).astype(int)
tab["share caught"] = (tab["caught"] / tab["all cases"] * 100).round(1)
tab.index.name = "quarters before the breach"

print(f"{int(lead['caught'].sum())} of {len(lead)} troubled bank-quarters caught "
      f"at the 1% budget\n")
print(tab.to_string())

# per bank: the earliest quarter at which the model had it on the list
per_bank = lead[lead["caught"]].groupby("CERT")["lead_q"].max()
print(f"\nEarliest warning per bank, {len(per_bank)} of {lead['CERT'].nunique()} "
      f"troubled banks reached:")
print(per_bank.value_counts().sort_index(ascending=False)
      .rename("banks").rename_axis("quarters of warning").to_string())

104 of 160 troubled bank-quarters caught at the 1% budget

                            all cases  caught  share caught
quarters before the breach                                 
1                                  48      34          70.8
2                                  40      28          70.0
3                                  38      22          57.9
4                                  34      20          58.8

Earliest warning per bank, 33 of 44 troubled banks reached:
quarters of warning
4    20
3     2
2     5
1     6


In [24]:
# The 44 troubled banks by name: how many of their warning quarters the model
# reached, the earliest warning it gave, and which of them went on to fail.
g = lead.groupby("CERT")
detail = pd.DataFrame({"warning_qtrs": g.size(), "caught_qtrs": g["caught"].sum()})
detail["earliest_caught"] = lead[lead["caught"]].groupby("CERT")["lead_q"].max()
detail["NAME"] = test_scored.groupby("CERT")["NAME"].last()

fails = pd.read_parquet(PROCESSED.parent / "raw" / "failures.parquet")
fails["FAILDATE"] = pd.to_datetime(fails["FAILDATE"], errors="coerce")
fails = (fails[fails["RESTYPE"] == "FAILURE"].dropna(subset=["CERT"])
         .assign(CERT=lambda d: d["CERT"].astype("int64"))
         .sort_values("FAILDATE").drop_duplicates("CERT", keep="last"))
detail["FAILED"] = detail.index.map(fails.set_index("CERT")["FAILDATE"])

out = detail[["NAME", "warning_qtrs", "caught_qtrs", "earliest_caught", "FAILED"]]
failed = out[out["FAILED"].notna()].sort_values("FAILED")

print(f"{len(out)} troubled banks | reached by the model: "
      f"{int((out['caught_qtrs'] > 0).sum())} | later failed: {len(failed)}\n")
print("THE ONES THAT FAILED")
print(failed.to_string())
print(f"\nof the {len(failed)} that failed, the model reached "
      f"{int((failed['caught_qtrs'] > 0).sum())}, "
      f"{int((failed['earliest_caught'] == 4).sum())} of them a full year ahead")

44 troubled banks | reached by the model: 33 | later failed: 10

THE ONES THAT FAILED
                               NAME  warning_qtrs  caught_qtrs  earliest_caught     FAILED
CERT                                                                                      
17719  FARMERS&MERCHANTS STB ARGONI             1            1              1.0 2017-10-13
58112         LOUISA COMMUNITY BANK             4            4              4.0 2019-10-25
58317                 RESOLUTE BANK             4            4              4.0 2019-10-25
21111         CITY NB OF NEW JERSEY             3            3              2.0 2019-11-01
18265            ERICSON STATE BANK             4            4              4.0 2020-02-14
14361              FIRST STATE BANK             4            4              4.0 2020-04-03
15426             ALMENA STATE BANK             4            4              4.0 2020-10-23
8758                  CITIZENS BANK             4            2              2.0 2023-11-03
5748

### Reading the ladder

There is no correct alert budget. It is set by how many banks supervisors can actually examine,
which is a resourcing question, not a modelling one, so performance is reported across a range
rather than at one point, following Cole & White (2012).

**Random is included as the floor.** It catches whatever share of banks it flags: 1% of banks
gives 1% of cases. Every other row has to beat that to mean anything, and the distance above it
is what the lift column in the summary measures.

Three things the ladder makes visible that a single number hides:

- **The ranking of models changes with the budget.** Nothing about the models changes; only the
  willingness to look does. A comparison quoted at one budget is a comparison at one arbitrary
  point.
- **Type I and Type II move in opposite directions.** Flagging more banks catches more cases and
  wastes more examiner time. There is no setting that improves both.
- **False-alarm rates stay high everywhere, and that is arithmetic rather than weakness.** With
  160 cases among 161,117 bank-quarters, flagging 1% means 1,611 alerts for at most 160 possible
  hits, a perfect model would still be wrong 90% of the time. Judge the catch rate at a
  budget, not the false-alarm rate in isolation.

---

# Summary

Everything above, in one table and four conclusions.

In [25]:
ALERT_BUDGET = 0.01     # reporting convention; see the ladder above


def caught_at(scores, y, budget=ALERT_BUDGET):
    k = int(len(scores) * budget)
    return y[np.argsort(-np.asarray(scores, float))[:k]].sum() / y.sum()


scored_by = {
    "naive (flag everything)": np.random.RandomState(0).rand(len(y_test_scored)),
    "capital headroom": headroom_scored,
    "logistic regression (alternative)":
        logistic.predict_proba(X_test_scored[LR_FEATURES])[:, 1],
    "gradient boosting (champion)": boosting.predict_proba(X_test_scored)[:, 1],
}
kind = {"naive (flag everything)": "benchmark", "capital headroom": "benchmark"}

rows = []
for name, s in scored_by.items():
    r = evaluate(name, s, y_test_scored)
    r["kind"] = kind.get(name, "method")
    rows.append(r)

for family in ["MLP", "GRU"]:
    block = deep[deep["family"] == family]
    rows.append({"kind": "method", "model": f"{family} (3-seed mean)",
                 "ROC-AUC": block["ROC-AUC"].mean(), "PR-AUC": block["PR-AUC"].mean(),
                 "lift": block["lift"].mean()})

# Only threshold-free metrics here. Anything measured at an alert budget lives in the
# ladder above, where the budget is visible rather than buried in a column header.
summary = (pd.DataFrame(rows)[["kind", "model", "ROC-AUC", "PR-AUC", "lift"]]
           .sort_values("kind", ascending=False, kind="stable"))

print(f"TEST 2017Q1-2025Q1 | {len(test_scored):,} rows | {int(y_test_scored.sum())} cases "
      f"| base rate {y_test_scored.mean():.4f}\n")
print(summary.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nEvery row is measured on the same rows, the same event, and the same test set.")
print("SCOR is not included: it predicts a different event on a different population --")
print("see the context section above.")

TEST 2017Q1-2025Q1 | 161,117 rows | 160 cases | base rate 0.0010

     kind                             model  ROC-AUC  PR-AUC     lift
   method logistic regression (alternative)   0.9001  0.0307  30.8945
   method      gradient boosting (champion)   0.9190  0.2194 220.9655
   method                 MLP (3-seed mean)   0.8831  0.0855  86.1434
   method                 GRU (3-seed mean)   0.9094  0.1217 122.5182
benchmark           naive (flag everything)   0.5378  0.0012   1.1642
benchmark                  capital headroom   0.8682  0.1282 129.1116

Every row is measured on the same rows, the same event, and the same test set.
SCOR is not included: it predicts a different event on a different population --
see the context section above.


## Is the gap real, or is it 160 cases of luck?

Gradient boosting scores 0.219 against the capital benchmark's 0.128. That looks like a
clear win. But every figure in this notebook now rests on **160 cases**, and a different 160
would have given different numbers. Nothing so far says whether the gap survives that.

The standard answer is a **bootstrap**: resample the test set with replacement several
hundred times, recompute the metric on each resample, and look at how much it moves.
Petropoulos et al. (2020) do exactly this, for the same reason.

One refinement matters here: the resampling unit is the **bank**, not the row. A troubled
bank contributes up to four warning quarters that the models catch or miss together, so
resampling rows would count four correlated quarters as four independent pieces of evidence
and make every interval look narrower than it is.

Two things get measured:

- **A range for each model**, how much its own score bounces.
- **The paired difference**, the same resample scored by both models, subtracted. This is
  the one that settles the question, because it cancels the shared noise: if one draw
  happens to contain easier cases, both models benefit, and the *difference* is unaffected.
  If the 95% interval for that difference stays above zero, the gap is real.

In [26]:
N_BOOT = 500
rng = np.random.RandomState(0)

boot_scores = {
    "capital headroom": headroom_scored,
    "logistic regression": logistic.predict_proba(X_test_scored[LR_FEATURES])[:, 1],
    "gradient boosting": boosting.predict_proba(X_test_scored)[:, 1],
    "MLP": np.mean(dl_scores["MLP"], axis=0),
    "GRU": np.mean(dl_scores["GRU"], axis=0),
}

# Cluster bootstrap: resample BANKS, not rows. A troubled bank contributes up to four
# warning quarters that succeed or fail together, so row resampling would treat four
# correlated quarters as four independent pieces of evidence and understate the noise.
bank_codes, _ = pd.factorize(test_scored["CERT"].values)
order_by_bank = np.argsort(bank_codes, kind="stable")
counts = np.bincount(bank_codes)
rows_by_bank = np.split(order_by_bank, np.cumsum(counts)[:-1])
n_banks = len(rows_by_bank)

draws = {name: [] for name in boot_scores}
caught_draws = {(name, b): [] for name in boot_scores for b in BUDGETS}
n = len(y_test_scored)

for _ in range(N_BOOT):
    pick = rng.randint(0, n_banks, n_banks)    # same resampled banks for every model
    idx = np.concatenate([rows_by_bank[j] for j in pick])
    y_b = y_test_scored[idx]
    if y_b.sum() == 0:
        continue
    k_at = {b: int(len(idx) * b) for b in BUDGETS}
    for name, s in boot_scores.items():
        s_b = np.asarray(s)[idx]
        draws[name].append(average_precision_score(y_b, s_b))
        order = np.argsort(-s_b)
        for b in BUDGETS:                      # catch rate at each rung of the ladder
            caught_draws[(name, b)].append(y_b[order[:k_at[b]]].sum() / y_b.sum())

draws = {k: np.array(v) for k, v in draws.items()}
caught_draws = {k: np.array(v) for k, v in caught_draws.items()}

intervals = pd.DataFrame([
    {"model": name,
     "PR-AUC": average_precision_score(y_test_scored, boot_scores[name]),
     "2.5%": np.percentile(v, 2.5),
     "97.5%": np.percentile(v, 97.5),
     "width": np.percentile(v, 97.5) - np.percentile(v, 2.5)}
    for name, v in draws.items()
])
print(f"PR-AUC with 95% intervals ({N_BOOT} bank-cluster resamples, {n_banks:,} banks)\n")
print(intervals.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\n\nPAIRED DIFFERENCE vs the capital benchmark\n")
base = draws["capital headroom"]
comparisons = pd.DataFrame([
    {"model": name,
     "mean gap": (v - base).mean(),
     "2.5%": np.percentile(v - base, 2.5),
     "97.5%": np.percentile(v - base, 97.5),
     "beats benchmark": f"{(v > base).mean():.0%} of resamples"}
    for name, v in draws.items() if name != "capital headroom"
])
print(comparisons.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))


PR-AUC with 95% intervals (500 bank-cluster resamples, 5,738 banks)

              model  PR-AUC   2.5%  97.5%  width
   capital headroom  0.1282 0.0692 0.2187 0.1495
logistic regression  0.0307 0.0157 0.0580 0.0423
  gradient boosting  0.2194 0.1408 0.3306 0.1898
                MLP  0.1185 0.0597 0.2049 0.1452
                GRU  0.1464 0.0659 0.2371 0.1711


PAIRED DIFFERENCE vs the capital benchmark

              model  mean gap    2.5%   97.5%   beats benchmark
logistic regression   -0.1050 -0.1807 -0.0353   0% of resamples
  gradient boosting   +0.0906 +0.0222 +0.1642 100% of resamples
                MLP   -0.0140 -0.0923 +0.0644  36% of resamples
                GRU   +0.0154 -0.0733 +0.1033  65% of resamples


### How to read this

**The individual intervals will be wide.** That is the honest consequence of 160 cases, and it
is why no single PR-AUC in this notebook should be quoted to three decimal places as though it
were precise.

**The paired difference is the number that decides it.** If its 95% interval includes zero, the
apparent win is inside the noise and should be reported as "comparable to the benchmark", not as
an improvement. If it stays above zero, the gap is real even though the individual intervals
overlap, overlapping intervals do *not* mean two models are indistinguishable, which is a
common misreading.

The "beats benchmark" column gives the same thing in plainer terms: the share of resampled test
sets in which the model came out ahead. Close to 50% means a coin flip.

### And the same test on the catch rates

PR-AUC summarises the whole ranking, but the ladder showed the models separating sharply at
practical alert budgets. Those gaps deserve the same scrutiny, an apparent 15-point
advantage is worth nothing if it is inside the sampling noise of 160 cases.

Same 500 resamples, same paired logic: each draw is scored by every model, and the
difference against the capital benchmark is taken within the draw.

In [27]:
gaps = []
for b in BUDGETS:
    base = caught_draws[("capital headroom", b)]
    for name in boot_scores:
        if name == "capital headroom":
            continue
        d = caught_draws[(name, b)] - base
        gaps.append({
            "alert budget": f"{b:.1%}", "model": name,
            "gap": d.mean(),
            "2.5%": np.percentile(d, 2.5),
            "97.5%": np.percentile(d, 97.5),
            "beats benchmark": (d > 0).mean(),
            "significant": "yes" if np.percentile(d, 2.5) > 0 else "no",
        })

gaps = pd.DataFrame(gaps)
print("CATCH RATE (troubled bank-quarters) vs the capital benchmark, in percentage points\n")
for b in BUDGETS:
    block = gaps[gaps["alert budget"] == f"{b:.1%}"]
    print(f"--- {b:.1%} alert budget ({int(len(y_test_scored) * b):,} filings) ---")
    print(block[["model", "gap", "2.5%", "97.5%", "beats benchmark", "significant"]]
          .to_string(index=False,
                     formatters={"gap": lambda v: f"{v:+.1%}",
                                 "2.5%": lambda v: f"{v:+.1%}",
                                 "97.5%": lambda v: f"{v:+.1%}",
                                 "beats benchmark": lambda v: f"{v:.0%}"}))
    print()

CATCH RATE (troubled bank-quarters) vs the capital benchmark, in percentage points

--- 0.1% alert budget (161 filings) ---
              model    gap   2.5%  97.5% beats benchmark significant
logistic regression -18.8% -28.9%  -7.5%              0%          no
  gradient boosting  +6.4%  -1.3% +14.2%             94%          no
                MLP  -3.3% -13.5%  +8.3%             24%          no
                GRU  -3.0% -13.2%  +7.9%             25%          no

--- 0.2% alert budget (322 filings) ---
              model    gap   2.5%  97.5% beats benchmark significant
logistic regression -20.1% -32.2%  -6.8%              1%          no
  gradient boosting  +8.6%  -0.5% +17.7%             96%          no
                MLP  -3.5% -15.7%  +8.9%             29%          no
                GRU  -1.2% -14.2% +12.0%             42%          no

--- 0.3% alert budget (483 filings) ---
              model    gap   2.5%  97.5% beats benchmark significant
logistic regression -21.7% -36.5%  

### What the two tests together say

For the champion, they agree. The PR-AUC gap over the benchmark is +0.091 with a 95%
interval of +0.022 to +0.164, ahead in 100% of resamples, and the catch-rate gaps are
positive and significant at every budget from 1% through 14%, +15 points at the 1%
budget alone. Past 15% the interval starts touching zero: at wide budgets the benchmark
catches up, and the honest claim narrows to the budgets a supervisor would actually run. The win
holds on the whole ranking and at the operating points a supervisor would actually run.

The same tests are just as decisive in the other direction for logistic regression: it is
*behind* the benchmark on PR-AUC in every resample, and behind on catch rate at tight
budgets. Interpretability is its remaining argument, not accuracy.

**What this means for the writeup.** The defensible claim is now the strong one: *gradient
boosting beats ranking by capital ratio, the supervisor's status quo, and the bootstrap says
the gap is real on both the global ranking and the catch rates.* The honest qualifiers are
the single weak test year (2022) and the small case count behind every number.

## Five conclusions

**1. The champion beats the status quo, and the gap survives the bootstrap.**
Gradient boosting: ROC-AUC 0.919, PR-AUC 0.219 against the capital benchmark's 0.128, lift
221x on a 0.10% base rate. The paired bank-cluster bootstrap puts the PR-AUC gap at +0.022 to +0.164, ahead in
100% of resamples, and the catch-rate advantage (+15 points at the 1% budget) is
significant at every budget from 1% through 14%.

**2. The mechanism story shrank from three bad years to one.**
Scored year by year, the model holds 0.92–0.98 everywhere except 2022 (0.72, on 17 cases).
On the 2008 crisis, trained only through 2005, it wins outright: PR-AUC 0.421 against 0.298.
The rate-shock mechanism is still the model's weakest ground, but it is a limitation now,
not a collapse.

**3. The label definition was the biggest modelling decision in the project.**
The critical-undercapitalization test originally used book equity (`EQV`), which absorbs
unrealized bond losses that PCA's Tier 1 definition excludes. In a rate shock the two
diverge: 478 bank-quarters landed in the worst tier while passing every capital ratio, and
they generated most of the old test cases. Moving the test to Tier 1 tangible equity
(12 CFR 324.403) cut the test base rate from 0.35% to 0.10%, and reversed the Week 4 model
choice, which had been made on the artifact. Book-equity insolvency stays in the data as a
feature (`econ_insolvent`), where it earns its place.

**4. Deep learning does not earn the slot, and seed variance is why single runs mislead.**
The GRU lands just under the untrained benchmark, the MLP under both, and both swing by
roughly half their mean PR-AUC across three seeds. Petropoulos et al.'s result, neural
networks competitive on this task, does not replicate here at this base rate.

**5. Two inherited assumptions did not survive testing.**
The FRED macro block contributes nothing on the modern period (slightly negative, in fact)
and about 8% of PR-AUC on the 2008 crisis, replicating Nuxoll (2003). And the merged-away
banks are normal consolidation, not hidden distress: they were gaining capital when they
left the panel.